In [7]:
import os, sys, numpy
from typing import Any
from pandas import Series, DataFrame, Timestamp, Timedelta
from pandas import read_pickle, read_csv, concat
from pandas import Index, MultiIndex, DatetimeIndex
from tqdm import tqdm

tests = dict[str, Any]()
for fn in os.listdir("../tests/"):
    if fn.endswith(".pickle"):
        key = fn[: 14]
        if key not in tests:
            tests[key] = list[str]()
        list.append(tests[key], fn)

print("Available tests:")
for key, files in tests.items():
    print(f" >> {key}: ({len(files)} files) ... {files}")
file_keys = ["ticks_p", "candles_p", "ticks_b", "candles_b"]

Available tests:


In [ ]:
side = "b"
columns = Index([*"ohlc"])
stats = dict()
ccount = dict()
iterator = tqdm(tests.items())
for key, files in iterator:
    ccount[key] = dict()
    iterator.set_description(f"Processing {key}...")
    dft = read_pickle(f"../tests/{key}_ticks_b.pickle")
    dfc = read_pickle(f"../tests/{key}_candles_b.pickle")
    dfp = read_pickle(f"../tests/{key}_candles_p.pickle")
    dft = dft.reset_index(["venue"], drop = True)
    dfc = dfc.reset_index(["venue"], drop = True)
    dfp = dfp.reset_index(["venue"], drop = True)
    dft = dft.reset_index("time")
    dft["time"] = dft["time"].dt.floor("1s")
    dft = dft.groupby(["symbol", "time"])["p" + side].apply(list)
    dfc = dfc[[*(columns + side), "volume"]]
    dfp = dfp[[*(columns + side), "volume"]]
    dfc.columns, dfp.columns = [*columns, "v"], [*columns, "v"]
    dfc = dfc.rename_axis("field", axis = "columns")
    dfp = dfp.rename_axis("field", axis = "columns")
    
    for tf in dfp.index.get_level_values("tf").unique():
        nc = dfc.query("tf == @tf").shape[0]
        np = dfp.query("tf == @tf").shape[0]
        ccount[key][tf] = {"b": nc, "p": np, "dt": Timedelta(
                tf[1 :] + tf[0].lower().replace("m", "min"))}
    dfc = dfc.xs("S1", level = "tf", drop_level = True)
    dfp = dfp.xs("S1", level = "tf", drop_level = True)

    comp = DataFrame.merge(dfc.stack("field").rename("b"), dfp.stack("field").rename("p"),
            left_index = True, right_index = True, how = "outer").sort_index().dropna()
    stat: Series = comp["b"] / comp["p"] - 1
    stat = stat.groupby(["symbol", "field"]).describe()
    stats[key] = stat

stats = concat(stats, names = ["test", "symbol", "field"])
ccount = DataFrame.from_dict(ccount, orient = "index")
ccount = ccount.stack().apply(Series)
ccount = ccount.rename_axis(["key", "tf"])
ccount["diff"] = ccount["p"] - ccount["b"]
ccount["pat"] = ccount.apply("{0[b]}/{0[p]}".format, axis = "columns")
# ccount["pat"].unstack("tf").sort_index().transpose()

In [114]:
key = "20260513_061634-20260513_061829"
dft = read_csv(f"../logs/{key}_ticks.csv")
dfc = read_csv(f"../logs/{key}_candles.csv")
dfc = dfc.drop(columns = ["venue", "symbol"]).set_index("time")
dft = dft.drop(columns = ["venue", "symbol"]).set_index("time")
dfc.index = DatetimeIndex(dfc.index)
dft.index = DatetimeIndex(dft.index)
tfs = dict.fromkeys(dfc["tf"].unique())
for tf in tfs:
    tfd = tf[1:] + " " + str.lower(tf[0])
    tfs[tf] = Timedelta(str.replace(tfd, "m", "min"))
dfc = dfc.set_index("tf", append = True).swaplevel()
dfc = dfc.sort_index()
dfp = dict.fromkeys(tfs)
for tf, tfd in tfs.items():
    df = dft.copy()[["pa", "pb"]]
    df.index = df.index.floor(tfd)
    dfv = df.groupby("time")["pa"].count().rename("volume")
    df = df.groupby("time").ohlc()
    df.columns = df.columns.map(lambda c: c[1][0] + c[0][-1])
    df = DataFrame.merge(dfv, df, how = "outer",
        left_index = True, right_index = True)
    dfp[tf] = df

dfp = concat(dfp, names = ["tf", "time"]).sort_index()
dfc = dfc.rename_axis("field", axis = "columns").sort_index().stack("field").rename("b")
dfp = dfp.rename_axis("field", axis = "columns").sort_index().stack("field").rename("p")
df = DataFrame.merge(dfc, dfp, how = "outer", left_index = True, right_index = True)
df = df.dropna(subset = ["b"])
df: Series = (df["b"] - df["p"]).abs()
stats = df.groupby(["field", "tf"]).describe()
print(stats.to_string())

            count        mean         std    min     25%    50%     75%     max
field  tf                                                                      
ca     M1     1.0    0.000000         NaN    0.0    0.00    0.0    0.00     0.0
       M2     1.0    0.000000         NaN    0.0    0.00    0.0    0.00     0.0
       M3     1.0    0.000000         NaN    0.0    0.00    0.0    0.00     0.0
       M6     1.0    0.000000         NaN    0.0    0.00    0.0    0.00     0.0
       S1    82.0    0.000000    0.000000    0.0    0.00    0.0    0.00     0.0
       S10    8.0    0.000000    0.000000    0.0    0.00    0.0    0.00     0.0
       S12    7.0    0.000000    0.000000    0.0    0.00    0.0    0.00     0.0
       S15    5.0    0.000000    0.000000    0.0    0.00    0.0    0.00     0.0
       S2    41.0    0.000000    0.000000    0.0    0.00    0.0    0.00     0.0
       S20    4.0    0.000000    0.000000    0.0    0.00    0.0    0.00     0.0
       S3    27.0    0.000000    0.00000